# Network Protocol Simulator: ARP and ICMP Logic
### Objective
This project simulates the communication between two nodes in a local network. It demonstrates the encapsulation process:
1. **ARP (Address Resolution Protocol):** Finding the hardware address (MAC) from an IP.
2. **ICMP (Internet Control Message Protocol):** Simulating a "Ping" request/reply.

### Engineering Concepts
- **Packet Encapsulation:** Nesting protocols within layers.
- **State Machines:** Managing the "Wait" and "Response" states of a network node.

In [1]:
class NetworkNode:
    def __init__(self, name, ip, mac):
        self.name = name
        self.ip = ip
        self.mac = mac
        self.arp_table = {}  # Cache: {IP: MAC}

    def send_arp_request(self, target_ip, network_nodes):
        print(f"[{self.name}] Broadcasting ARP Request: Who has {target_ip}?")
        for node in network_nodes:
            if node.ip == target_ip:
                print(f"[{node.name}] I have {target_ip}. My MAC is {node.mac}")
                self.arp_table[target_ip] = node.mac
                return True
        return False

    def ping(self, target_ip, network_nodes):
        # Step 1: Check ARP Table
        if target_ip not in self.arp_table:
            success = self.send_arp_request(target_ip, network_nodes)
            if not success:
                print(f"[{self.name}] Ping failed: Host unreachable.")
                return

        # Step 2: Send ICMP Echo Request
        target_mac = self.arp_table[target_ip]
        print(f"[{self.name}] Sending ICMP Echo Request to {target_ip} at {target_mac}")
        print(f"[{self.name}] Ping Success: Reply received!")

In [2]:
# Create nodes based on a typical LAN setup
laptop = NetworkNode("Imran-Laptop", "192.168.1.5", "AA:BB:CC:DD:EE:01")
server = NetworkNode("Polytech-Server", "192.168.1.10", "11:22:33:44:55:02")
router = NetworkNode("Gateway-Router", "192.168.1.1", "FF:FF:FF:FF:FF:FE")

network = [laptop, server, router]

# Simulate the first communication (ARP will trigger)
print("--- FIRST PING (ARP Cache Empty) ---")
laptop.ping("192.168.1.10", network)

print("\n--- SECOND PING (ARP Cache Filled) ---")
# This shows the efficiency of the cache logic
laptop.ping("192.168.1.10", network)

--- FIRST PING (ARP Cache Empty) ---
[Imran-Laptop] Broadcasting ARP Request: Who has 192.168.1.10?
[Polytech-Server] I have 192.168.1.10. My MAC is 11:22:33:44:55:02
[Imran-Laptop] Sending ICMP Echo Request to 192.168.1.10 at 11:22:33:44:55:02
[Imran-Laptop] Ping Success: Reply received!

--- SECOND PING (ARP Cache Filled) ---
[Imran-Laptop] Sending ICMP Echo Request to 192.168.1.10 at 11:22:33:44:55:02
[Imran-Laptop] Ping Success: Reply received!


## Performance Analysis: ARP Overhead
In a real network, the first packet takes longer because of the ARP broadcast. 
We can model the total time $T$ as:
$$T_{total} = T_{arp} + T_{icmp}$$
Where $T_{arp}$ only occurs if the IP is not in the cache.

### Script that adds a graphical window with moving "packets"

In [5]:
import tkinter as tk
import time

class NetworkVisualizer:
    def __init__(self, root):
        self.root = root
        self.root.title("Polytech Network Simulator - Packet Tracer")
        self.canvas = tk.Canvas(root, width=600, height=400, bg="#f0f0f0")
        self.canvas.pack(pady=20)

        # 1. Define Node Positions (Coordinates)
        self.nodes = {
            "Imran-Laptop": (100, 200, "#3498db"),  # Blue
            "Server": (500, 200, "#e74c3c")         # Red
        }

        # 2. Draw Nodes and Labels
        for name, (x, y, color) in self.nodes.items():
            self.canvas.create_oval(x-30, y-30, x+30, y+30, fill=color, outline="black", width=2)
            self.canvas.create_text(x, y+50, text=name, font=("Arial", 12, "bold"))

        # 3. Add Control Button
        self.btn = tk.Button(root, text="Send ICMP Packet (Ping)", command=self.animate_packet, 
                             bg="#2ecc71", fg="white", font=("Arial", 10, "bold"))
        self.btn.pack(pady=10)

        # Log area (Simulating Terminal)
        self.log = tk.Label(root, text="System Ready", fg="blue", font=("Courier", 10))
        self.log.pack()

    def animate_packet(self):
        self.btn.config(state="disabled")
        self.log.config(text="Sending ARP Request...", fg="orange")
        
        # Create Packet (Small Rectangle)
        packet = self.canvas.create_rectangle(110, 190, 140, 210, fill="yellow", outline="black")
        
        # Movement Logic (X-axis)
        start_x = 110
        end_x = 460
        
        for x in range(start_x, end_x, 5):
            self.canvas.move(packet, 5, 0)
            self.root.update()
            time.sleep(0.01) # Controls speed

        self.log.config(text="ICMP Echo Reply Received!", fg="green")
        
        # Remove packet after arrival
        time.sleep(0.5)
        self.canvas.delete(packet)
        self.btn.config(state="normal")

# Run the Application
if __name__ == "__main__":
    root = tk.Tk()
    app = NetworkVisualizer(root)
    root.mainloop()